In [9]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [1]:
# Install the Elasticsearch client
!pip install -q elasticsearch
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.6/993.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.5 MB/s eta 0:00:00


In [11]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
import json
from tqdm import tqdm
import pandas as pd
import torch

# connecting to elasticsearch

In [2]:
from elasticsearch import Elasticsearch


es = Elasticsearch(
    "https://my-elasticsearch-project-cb62c1.es.us-central1.gcp.elastic.cloud:443",
    api_key="T0g4UW9aNEJOaGxPa3ZuYy1meEo6MmpkenZkRXBmYXRJcThOdl82MlQ4QQ==",
    request_timeout=60,           # ← This is the correct parameter
    retry_on_timeout=True
)

# Test the connection
try:
    info = es.info()
    print("✅ Successfully connected to Elasticsearch Cloud!")
    print(f"Version: {info['version']['number']}")
    print(f"Cluster: {info['cluster_name']}")
except Exception as e:
    print("❌ Connection failed:", str(e))

✅ Successfully connected to Elasticsearch Cloud!
Version: 9.5.0
Cluster: cb62c117b6fa49f8a9fc0fe585e94ade


# Create Index

In [3]:
index_name = "miracl_long_docs_chunked"

mapping = {
    "mappings": {
        "properties": {
            "doc_id":        {"type": "keyword"},
            "chunk_text":    {"type": "text", "analyzer": "standard"},
            "chunk_embedding":{"type": "dense_vector", "dims": 768, "index": True, "similarity": "cosine"},
            "original_doc_length": {"type": "integer"},
            "chunk_id":      {"type": "integer"}
        }
    }
}

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

es.indices.create(index=index_name, body=mapping)
print(f"Index '{index_name}' created.\n")

Index 'miracl_long_docs_chunked' created.



# Indexing Documents

In [5]:
# ===================== CONFIG =====================
corpus_file = "/content/long_docs.jsonl"
index_name = "miracl_long_docs_chunked"
chunk_size = 600
chunk_overlap = 90

# ===================== LOAD MODEL =====================
print("Loading Jina v5 Nano...")
model = SentenceTransformer(
    "jinaai/jina-embeddings-v5-text-nano-retrieval",
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING + INDEXING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

actions = []
batch_size = 40
total_chunks = 0
total_docs = 0

indexing_start = time.time()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Processing")):
        if not line.strip():
            continue

        doc = json.loads(line)
        text = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            embedding = model.encode([chunk_text], normalize_embeddings=True)[0]

            document = {
                "doc_id": doc_id,
                "chunk_text": chunk_text,
                "chunk_embedding": embedding.tolist(),
                "original_doc_length": len(text),
                "chunk_id": chunk_idx
            }

            # v8 bulk format: metadata line + document line
            actions.append({"index": {"_index": index_name}})
            actions.append(document)

            total_chunks += 1

            if len(actions) >= batch_size * 2:  # *2 because each chunk = 2 items
                es.bulk(operations=actions)
                actions.clear()

# Final batch
if actions:
    es.bulk(operations=actions)

indexing_elapsed = time.time() - indexing_start

# ===================== INDEXING TIME =====================
print(f"""
============== INDEXING TIME ==============
  Total time:         {indexing_elapsed:.1f} seconds ({indexing_elapsed/60:.2f} min)
  Total docs:         {total_docs}
  Total chunks:       {total_chunks}
  Chunks per second:  {total_chunks/indexing_elapsed:.1f}
  Avg per chunk:      {indexing_elapsed/total_chunks*1000:.2f} ms
===========================================
""")

# ===================== STORAGE SIZE =====================
es.indices.refresh(index=index_name)
time.sleep(2)

stats    = es.indices.stats(index=index_name)
store    = stats["indices"][index_name]["total"]["store"]
size_bytes = store["size_in_bytes"]
size_mb    = size_bytes / (1024 ** 2)
size_gb    = size_bytes / (1024 ** 3)
doc_count  = stats["indices"][index_name]["total"]["docs"]["count"]

print(f"""
============== STORAGE SIZE ==============
  Index name:         {index_name}
  Total documents:    {doc_count} chunks
  Index size:         {size_bytes:,} bytes
  Index size:         {size_mb:.2f} MB
  Index size:         {size_gb:.3f} GB
  Avg per chunk:      {size_bytes/doc_count/1024:.2f} KB
==========================================
""")

print(f"Indexing completed! Total chunks indexed: {total_chunks}")

Loading Jina v5 Nano...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!



Processing: 0it [00:00, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Processing: 68611it [1:18:22, 14.59it/s]



============== INDEXING TIME ==============
  Total time:         4703.0 seconds (78.38 min)
  Total docs:         68611
  Total chunks:       165125
  Chunks per second:  35.1
  Avg per chunk:      28.48 ms



ApiError: ApiError(410, 'api_not_available_exception', 'Request for uri [/miracl_long_docs_chunked/_stats] with method [GET] exists but is not available when running in serverless mode')

In [6]:
# Fallback — just count docs
count = es.count(index=index_name)
print(f"Total indexed chunks: {count['count']}")

# Estimate storage size manually
embedding_dim = 128
bytes_per_float = 4  # float32
estimated_vector_bytes = total_chunks * embedding_dim * bytes_per_float
estimated_text_bytes = sum(len(c.encode('utf-8')) for c in [])  # rough

print(f"""
============== ESTIMATED STORAGE ==============
  Total chunks:              {total_chunks}
  Estimated vector storage:  {estimated_vector_bytes/1024**2:.2f} MB
  (actual may be 2-3x due to HNSW index overhead)
===============================================
""")

Total indexed chunks: 165125

============== ESTIMATED STORAGE ==============
  Total chunks:              165125
  Estimated vector storage:  80.63 MB
  (actual may be 2-3x due to HNSW index overhead)



# testing

In [14]:
# ===================== CONFIG =====================
corpus_file = "/content/long_docs.jsonl"
qrels_file  = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file = "/content/topics.miracl-v1.0-fa-dev.tsv"


# Reload filtered doc IDs from corpus
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))
print(f"Loaded {len(filtered_doc_ids)} doc IDs")

# Reload qrels
print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

# Reload topics
print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

# Build dataframes
topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"""
  Topics loaded:  {len(topics_df)}
  Qrels loaded:   {len(qrels_df)}
""")



Loading filtered doc IDs...


0it [00:00, ?it/s]

Loaded 68611 doc IDs
Loading qrels...


0it [00:00, ?it/s]

Loading topics...

  Topics loaded:  140
  Qrels loaded:   174



In [15]:
import time
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# ===================== CONFIG =====================
top_k = 10  # retrieve top 10 chunks

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...\n")

hits_at_5   = 0
hits_at_10  = 0
total_queries = 0
query_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    # Get relevant docs for this query from qrels
    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # ---- Query latency start ----
    query_start = time.time()

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

    # Search Elasticsearch
    response = es.search(
        index=index_name,
        body={
            "size": top_k,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.query_vector, 'chunk_embedding') + 1.0",
                        "params": {"query_vector": query_emb}
                    }
                }
            },
            "_source": ["doc_id"]
        }
    )

    # ---- Query latency end ----
    query_latency = (time.time() - query_start) * 1000  # ms
    query_latencies.append(query_latency)

    # Extract retrieved doc_ids
    hits = response["hits"]["hits"]
    retrieved_top10 = {str(h["_source"]["doc_id"]) for h in hits[:10]}
    retrieved_top5  = {str(h["_source"]["doc_id"]) for h in hits[:5]}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== METRICS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

latencies = np.array(query_latencies)

print(f"""
==================== RESULTS ====================
  Total queries evaluated:  {total_queries}
  Hit@5:                    {hit5:.4f}
  Hit@10:                   {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Mean:     {latencies.mean():.2f} ms
  Median:   {np.median(latencies):.2f} ms
  Min:      {latencies.min():.2f} ms
  Max:      {latencies.max():.2f} ms
  P90:      {np.percentile(latencies, 90):.2f} ms
  P95:      {np.percentile(latencies, 95):.2f} ms
  P99:      {np.percentile(latencies, 99):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "index_name":      index_name,
    "chunk_size":      chunk_size,
    "chunk_overlap":   chunk_overlap,
    "total_chunks":    total_chunks,
    "total_queries":   total_queries,
    "hit@5":           round(hit5, 4),
    "hit@10":          round(hit10, 4),
    "latency_mean_ms": round(latencies.mean(), 2),
    "latency_p95_ms":  round(np.percentile(latencies, 95), 2),
    "latency_p99_ms":  round(np.percentile(latencies, 99), 2),
}

print("Results summary:")
print(pd.DataFrame([results_summary]).to_string(index=False))

Evaluating 140 queries...



Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries evaluated:  140
  Hit@5:                    0.5929
  Hit@10:                   0.6786

============== QUERY LATENCY (ms) ==============
  Mean:     339.48 ms
  Median:   293.06 ms
  Min:      277.84 ms
  Max:      6059.97 ms
  P90:      314.34 ms
  P95:      319.83 ms
  P99:      600.04 ms

Results summary:
              index_name  chunk_size  chunk_overlap  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms
miracl_long_docs_chunked         600             90        165125            140 0.5929  0.6786           339.48          319.83          600.04


In [16]:
test_emb = model.encode(["test"], normalize_embeddings=True)
print(f"Embedding dim: {test_emb.shape[1]}")

Embedding dim: 768


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [17]:
# ===================== STEP 1: CHUNK ALL DOCS =====================
all_chunks_text = []
all_chunks_meta = []

with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue
        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        chunks = splitter.split_text(text)
        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"Total chunks to encode: {len(all_chunks_text)}")

# ===================== STEP 2: ENCODE IN BATCHES =====================
embeddings = model.encode(
    all_chunks_text,
    batch_size=128,        # ← tune this
    show_progress_bar=True,
    normalize_embeddings=True
)

# ===================== STEP 3: INDEX INTO ES =====================
actions = []
for i, (meta, emb) in enumerate(tqdm(zip(all_chunks_meta, embeddings), desc="Indexing", total=len(embeddings))):
    document = {
        "doc_id":              meta["doc_id"],
        "chunk_text":          all_chunks_text[i],
        "chunk_embedding":     emb.tolist(),
        "original_doc_length": meta["original_doc_length"],
        "chunk_id":            meta["chunk_id"]
    }
    actions.append({"index": {"_index": new_index}})
    actions.append(document)

    if len(actions) >= 40 * 2:
        es.bulk(operations=actions)
        actions.clear()

if actions:
    es.bulk(operations=actions)

Chunking: 0it [00:00, ?it/s]

Total chunks to encode: 165125


Batches:   0%|          | 0/1291 [00:00<?, ?it/s]

Indexing:   0%|          | 0/165125 [00:00<?, ?it/s]

NameError: name 'new_index' is not defined

In [18]:
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

# ===================== CONFIG =====================
corpus_file   = "/content/long_docs.jsonl"
new_index     = "miracl_long_docs_knn"
chunk_size    = 600
chunk_overlap = 90
batch_size    = 64        # tune based on your GPU VRAM
model_name    = "jinaai/jina-embeddings-v5-text-nano-retrieval"

# ===================== CONNECT TO ELASTICSEARCH =====================
es = Elasticsearch("https://my-elasticsearch-project-cb62c1.es.us-central1.gcp.elastic.cloud:443",
                   api_key="T0g4UW9aNEJOaGxPa3ZuYy1meEo6MmpkenZkRXBmYXRJcThOdl82MlQ4QQ")
print(f"Elasticsearch connected: {es.ping()}")

# ===================== CREATE INDEX WITH KNN MAPPING =====================
if es.indices.exists(index=new_index):
    es.indices.delete(index=new_index)
    print(f"Deleted existing index: {new_index}")

es.indices.create(index=new_index, body={
    "mappings": {
        "properties": {
            "chunk_embedding": {
                "type":       "dense_vector",
                "dims":       768,
                "index":      True,
                "similarity": "cosine"
            },
            "doc_id":              {"type": "keyword"},
            "chunk_text":          {"type": "text"},
            "original_doc_length": {"type": "integer"},
            "chunk_id":            {"type": "integer"}
        }
    }
})
print(f"Created index: {new_index}\n")

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks_text = []
all_chunks_meta = []
total_docs = 0

print("Chunking documents...")
with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue

        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"Total docs:   {total_docs}")
print(f"Total chunks: {len(all_chunks_text)}\n")

# ===================== ENCODE IN BATCHES =====================
print(f"Encoding {len(all_chunks_text)} chunks with batch_size={batch_size}...")
encode_start = time.time()

embeddings = model.encode(
    all_chunks_text,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)

encode_elapsed = time.time() - encode_start

print(f"""
============== ENCODING TIME ==============
  Total time:        {encode_elapsed:.1f}s ({encode_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/encode_elapsed:.1f}
  Embedding dim:     {embeddings.shape[1]}
  Dtype:             {embeddings.dtype}
  Matrix size:       {embeddings.nbytes/1024**2:.2f} MB
===========================================
""")

# ===================== INDEX INTO ELASTICSEARCH =====================
print("Indexing into Elasticsearch...")
index_start = time.time()
actions     = []
es_batch    = 40

for i, (meta, emb) in enumerate(tqdm(zip(all_chunks_meta, embeddings), desc="Indexing", total=len(embeddings))):
    document = {
        "doc_id":              meta["doc_id"],
        "chunk_text":          all_chunks_text[i],
        "chunk_embedding":     emb.tolist(),
        "original_doc_length": meta["original_doc_length"],
        "chunk_id":            meta["chunk_id"]
    }
    actions.append({"index": {"_index": new_index}})
    actions.append(document)

    if len(actions) >= es_batch * 2:
        es.bulk(operations=actions)
        actions.clear()

if actions:
    es.bulk(operations=actions)

index_elapsed = time.time() - index_start

print(f"""
============== INDEXING TIME ==============
  Total time:        {index_elapsed:.1f}s ({index_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/index_elapsed:.1f}
===========================================

============== TOTAL TIME =================
  Encoding + Indexing: {(encode_elapsed+index_elapsed)/60:.2f} min
===========================================
""")

# ===================== RELOAD TOPICS & QRELS =====================
qrels_file  = "/content/drive/MyDrive/qrels.miracl-v1.0-fa-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-fa-dev.tsv"

print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")
es.indices.refresh(index=new_index)

hits_at_5    = 0
hits_at_10   = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()
    encode_ms = (time.time() - t0) * 1000

    # KNN search
    t1 = time.time()
    response = es.search(
        index=new_index,
        body={
            "knn": {
                "field":          "chunk_embedding",
                "query_vector":   query_emb,
                "k":              10,
                "num_candidates": 100
            },
            "_source": ["doc_id"]
        }
    )
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    hits = response["hits"]["hits"]
    retrieved_top10 = {str(h["_source"]["doc_id"]) for h in hits[:10]}
    retrieved_top5  = {str(h["_source"]["doc_id"]) for h in hits[:5]}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat   = np.array(query_latencies)
elat  = np.array(encode_latencies)
slat  = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "index_name":        new_index,
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

Elasticsearch connected: True
Created index: miracl_long_docs_knn

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Chunking documents...


...

Indexing:   0%|          | 0/165125 [00:00<?, ?it/s]


============== INDEXING TIME ==============
  Total time:        1597.2s (26.62 min)
  Chunks per second: 103.4

============== TOTAL TIME =================
  Encoding + Indexing: 37.16 min

Loading filtered doc IDs...


Loading corpus IDs: 0it [00:00, ?it/s]

Loading qrels...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/qrels.miracl-v1.0-fa-dev.tsv'

In [19]:
# ===================== RELOAD TOPICS & QRELS =====================
qrels_file  = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file = "/content/topics.miracl-v1.0-fa-dev.tsv"

print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")
es.indices.refresh(index=new_index)

hits_at_5    = 0
hits_at_10   = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()
    encode_ms = (time.time() - t0) * 1000

    # KNN search
    t1 = time.time()
    response = es.search(
        index=new_index,
        body={
            "knn": {
                "field":          "chunk_embedding",
                "query_vector":   query_emb,
                "k":              10,
                "num_candidates": 100
            },
            "_source": ["doc_id"]
        }
    )
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    hits = response["hits"]["hits"]
    retrieved_top10 = {str(h["_source"]["doc_id"]) for h in hits[:10]}
    retrieved_top5  = {str(h["_source"]["doc_id"]) for h in hits[:5]}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat   = np.array(query_latencies)
elat  = np.array(encode_latencies)
slat  = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "index_name":        new_index,
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

Loading filtered doc IDs...


Loading corpus IDs: 0it [00:00, ?it/s]

Loading qrels...


Loading qrels: 0it [00:00, ?it/s]

Loading topics...
Topics: 140 | Qrels: 174

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.5143
  Hit@10:         0.6000

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    113.45 ms
    Median:  62.88 ms
    P95:     91.35 ms
    P99:     290.50 ms

  Encode only:
    Mean:    19.99 ms
    Median:  20.01 ms

  Search only:
    Mean:    93.46 ms
    Median:  42.05 ms

          index_name  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
miracl_long_docs_knn         600             90       68611        165125            140 0.5143     0.6           113.45           91.35           290.5           19.99           93.46


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [20]:
# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")
es.indices.refresh(index=new_index)

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()
    encode_ms = (time.time() - t0) * 1000

    # KNN search
    t1 = time.time()
    response = es.search(
        index=new_index,
        body={
            "knn": {
                "field":          "chunk_embedding",
                "query_vector":   query_emb,
                "k":              50,
                "num_candidates": 500
            },
            "_source": ["doc_id"]
        }
    )
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    hits = response["hits"]["hits"]
    retrieved_top10 = []
    seen = set()
    for h in hits:
        doc_id = str(h["_source"]["doc_id"])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "index_name":        new_index,
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.5214
  Hit@10:         0.6000

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    65.53 ms
    Median:  64.82 ms
    P95:     76.19 ms
    P99:     83.72 ms

  Encode only:
    Mean:    21.54 ms
    Median:  21.13 ms

  Search only:
    Mean:    43.99 ms
    Median:  43.54 ms

          index_name  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
miracl_long_docs_knn         600             90       68611        165125            140 0.5214     0.6            65.53           76.19           83.72           21.54           43.99


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API

# FAISS

In [25]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 73.1 MB/s eta 0:00:00


In [26]:
import faiss
print(faiss.__version__)

1.14.2


In [27]:
import time
import json
import torch
import numpy as np
import pandas as pd
import faiss
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# ===================== CONFIG =====================
corpus_file   = "/content/long_docs.jsonl"
chunk_size    = 600
chunk_overlap = 90
batch_size    = 64
model_name    = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qrels_file    = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file   = "/content/topics.miracl-v1.0-fa-dev.tsv"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks_text = []
all_chunks_meta = []
total_docs = 0

print("Chunking documents...")
with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue

        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"Total docs:   {total_docs}")
print(f"Total chunks: {len(all_chunks_text)}\n")

# ===================== ENCODE =====================
print(f"Encoding {len(all_chunks_text)} chunks with batch_size={batch_size}...")
encode_start = time.time()

embeddings = model.encode(
    all_chunks_text,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

# FAISS requires float32
embeddings = embeddings.astype('float32')

encode_elapsed = time.time() - encode_start

print(f"""
============== ENCODING ==============
  Total time:        {encode_elapsed:.1f}s ({encode_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/encode_elapsed:.1f}
  Embedding dim:     {embeddings.shape[1]}
  Matrix size:       {embeddings.nbytes/1024**2:.2f} MB
======================================
""")

# ===================== BUILD FAISS INDEX =====================
print("Building FAISS index...")
index_start = time.time()

dim = embeddings.shape[1]

# IndexFlatIP = exact inner product search (cosine since embeddings are normalized)
index = faiss.IndexFlatIP(dim)

# Use GPU if available
if torch.cuda.is_available():
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index)
    print("FAISS running on GPU")
else:
    print("FAISS running on CPU")

index.add(embeddings)
index_elapsed = time.time() - index_start

print(f"""
============== FAISS INDEX ==============
  Index type:    IndexFlatIP (exact search)
  Total vectors: {index.ntotal}
  Build time:    {index_elapsed:.2f}s
=========================================
""")

# Build chunk_to_doc mapping
chunk_to_doc = [meta["doc_id"] for meta in all_chunks_meta]

# ===================== LOAD TOPICS & QRELS =====================
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # FAISS search — retrieve more chunks for deduplication
    t1 = time.time()
    distances, indices = index.search(query_emb, k=50)
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for idx in indices[0]:
        if idx == -1:
            continue
        doc_id = str(chunk_to_doc[idx])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "FAISS-IndexFlatIP",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

# ===================== SAVE INDEX TO DRIVE =====================
# print("\nSaving FAISS index to Drive...")

# # Move to CPU before saving if on GPU
# if torch.cuda.is_available():
#     index_cpu = faiss.index_gpu_to_cpu(index)
# else:
#     index_cpu = index

# faiss.write_index(index_cpu, "/content/drive/MyDrive/faiss_index.bin")
# np.save("/content/drive/MyDrive/chunk_to_doc.npy", np.array(chunk_to_doc))
# np.save("/content/drive/MyDrive/embeddings.npy", embeddings)

# print("Saved: faiss_index.bin, chunk_to_doc.npy, embeddings.npy")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Chunking documents...


...

Total docs:   68611
Total chunks: 165125

Encoding 165125 chunks with batch_size=64...


Batches:   0%|          | 0/2581 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



============== ENCODING ==============
  Total time:        638.5s (10.64 min)
  Chunks per second: 258.6
  Embedding dim:     768
  Matrix size:       483.76 MB

Building FAISS index...


AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [28]:
# ===================== BUILD FAISS INDEX =====================
print("Building FAISS index...")
index_start = time.time()

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

index_elapsed = time.time() - index_start

print(f"""
============== FAISS INDEX ==============
  Index type:    IndexFlatIP (exact search)
  Total vectors: {index.ntotal}
  Build time:    {index_elapsed:.2f}s
=========================================
""")

# Build chunk_to_doc mapping
chunk_to_doc = [meta["doc_id"] for meta in all_chunks_meta]

# ===================== LOAD TOPICS & QRELS =====================
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # FAISS search
    t1 = time.time()
    distances, indices = index.search(query_emb, k=50)
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for idx in indices[0]:
        if idx == -1:
            continue
        doc_id = str(chunk_to_doc[idx])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "FAISS-IndexFlatIP",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

# ===================== SAVE INDEX TO DRIVE =====================
# print("\nSaving FAISS index to Drive...")
# faiss.write_index(index, "/content/drive/MyDrive/faiss_index.bin")
# np.save("/content/drive/MyDrive/chunk_to_doc.npy", np.array(chunk_to_doc))
# np.save("/content/drive/MyDrive/embeddings.npy", embeddings)
# print("Saved: faiss_index.bin, chunk_to_doc.npy, embeddings.npy")

Building FAISS index...

============== FAISS INDEX ==============
  Index type:    IndexFlatIP (exact search)
  Total vectors: 165125
  Build time:    0.34s

Loading filtered doc IDs...


Loading corpus IDs: 0it [00:00, ?it/s]

Loading qrels...


Loading qrels: 0it [00:00, ?it/s]

Loading topics...
Topics: 140 | Qrels: 174

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.6000
  Hit@10:         0.6929

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    65.66 ms
    Median:  64.17 ms
    P95:     78.13 ms
    P99:     91.65 ms

  Encode only:
    Mean:    21.71 ms
    Median:  21.45 ms

  Search only:
    Mean:    43.94 ms
    Median:  42.90 ms

     vector_store  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
FAISS-IndexFlatIP         600             90       68611        165125            140    0.6  0.6929            65.66           78.13           91.65           21.71           43.94


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API

# qdrant

In [29]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 12.2 MB/s eta 0:00:00


In [30]:
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# ===================== CONFIG =====================
corpus_file   = "/content/long_docs.jsonl"
chunk_size    = 600
chunk_overlap = 90
batch_size    = 64
model_name    = "jinaai/jina-embeddings-v5-text-nano-retrieval"
qrels_file    = "/content/qrels.miracl-v1.0-fa-dev.tsv"
topics_file   = "/content/topics.miracl-v1.0-fa-dev.tsv"
collection    = "miracl"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks_text = []
all_chunks_meta = []
total_docs = 0

print("Chunking documents...")
with open(corpus_file, 'r', encoding='utf-8') as f:
    for doc_idx, line in enumerate(tqdm(f, desc="Chunking")):
        if not line.strip():
            continue

        doc    = json.loads(line)
        text   = doc.get("text", "")
        doc_id = str(doc.get("docid") or doc.get("id") or doc_idx)

        if len(text) < 600:
            continue

        total_docs += 1
        chunks = splitter.split_text(text)

        for chunk_idx, chunk_text in enumerate(chunks):
            all_chunks_text.append(chunk_text)
            all_chunks_meta.append({
                "doc_id":              doc_id,
                "chunk_id":            chunk_idx,
                "original_doc_length": len(text)
            })

print(f"Total docs:   {total_docs}")
print(f"Total chunks: {len(all_chunks_text)}\n")

# ===================== ENCODE =====================
print(f"Encoding {len(all_chunks_text)} chunks with batch_size={batch_size}...")
encode_start = time.time()

embeddings = model.encode(
    all_chunks_text,
    batch_size=batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

encode_elapsed = time.time() - encode_start

print(f"""
============== ENCODING ==============
  Total time:        {encode_elapsed:.1f}s ({encode_elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks_text)/encode_elapsed:.1f}
  Embedding dim:     {embeddings.shape[1]}
  Matrix size:       {embeddings.nbytes/1024**2:.2f} MB
======================================
""")

# ===================== BUILD QDRANT INDEX =====================
print("Building Qdrant index...")
index_start = time.time()

# In-memory mode — no server needed
client = QdrantClient(":memory:")

client.create_collection(
    collection_name=collection,
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE
    )
)

# Upload in batches
upload_batch = 512
for i in tqdm(range(0, len(embeddings), upload_batch), desc="Uploading to Qdrant"):
    batch_end = min(i + upload_batch, len(embeddings))
    points = [
        PointStruct(
            id=j,
            vector=embeddings[j].tolist(),
            payload={"doc_id": all_chunks_meta[j]["doc_id"]}
        )
        for j in range(i, batch_end)
    ]
    client.upsert(collection_name=collection, points=points)

index_elapsed = time.time() - index_start

print(f"""
============== QDRANT INDEX ==============
  Collection:    {collection}
  Total vectors: {client.get_collection(collection).points_count}
  Build time:    {index_elapsed:.2f}s
==========================================
""")

# ===================== LOAD TOPICS & QRELS =====================
print("Loading filtered doc IDs...")
filtered_doc_ids = set()
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading corpus IDs"):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print("Loading qrels...")
qrels = {}
valid_queries = set()
with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading qrels"):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid, docid, rel = parts[0], parts[2], int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print("Loading topics...")
topics = {}
with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])
qrels_df  = pd.DataFrame([
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
])

print(f"Topics: {len(topics_df)} | Qrels: {len(qrels_df)}\n")

# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # Qdrant search
    t1 = time.time()
    results = client.search(
        collection_name=collection,
        query_vector=query_emb[0].tolist(),
        limit=50,
        with_payload=True
    )
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for r in results:
        doc_id = str(r.payload["doc_id"])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "Qdrant-InMemory",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Chunking documents...


...

Loading qrels: 0it [00:00, ?it/s]

Loading topics...
Topics: 140 | Qrels: 174

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


AttributeError: 'QdrantClient' object has no attribute 'search'

In [31]:
# ===================== EVALUATION =====================
print(f"Evaluating {len(topics_df)} queries...")

hits_at_5     = 0
hits_at_10    = 0
total_queries = 0
query_latencies  = []
encode_latencies = []
search_latencies = []

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    t0 = time.time()
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    encode_ms = (time.time() - t0) * 1000

    # Qdrant search (fixed for qdrant-client >= 1.7)
    t1 = time.time()
    results = client.query_points(
        collection_name=collection,
        query=query_emb[0].tolist(),
        limit=50,
        with_payload=True
    ).points
    search_ms = (time.time() - t1) * 1000

    total_ms = encode_ms + search_ms
    encode_latencies.append(encode_ms)
    search_latencies.append(search_ms)
    query_latencies.append(total_ms)

    # Deduplicate chunks → unique docs
    retrieved_top10 = []
    seen = set()
    for r in results:
        doc_id = str(r.payload["doc_id"])
        if doc_id not in seen:
            seen.add(doc_id)
            retrieved_top10.append(doc_id)
        if len(retrieved_top10) == 10:
            break

    retrieved_top5  = set(retrieved_top10[:5])
    retrieved_top10 = set(retrieved_top10)

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

# ===================== RESULTS =====================
hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

lat  = np.array(query_latencies)
elat = np.array(encode_latencies)
slat = np.array(search_latencies)

print(f"""
==================== RESULTS ====================
  Total queries:  {total_queries}
  Hit@5:          {hit5:.4f}
  Hit@10:         {hit10:.4f}
=================================================

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    {lat.mean():.2f} ms
    Median:  {np.median(lat):.2f} ms
    P95:     {np.percentile(lat, 95):.2f} ms
    P99:     {np.percentile(lat, 99):.2f} ms

  Encode only:
    Mean:    {elat.mean():.2f} ms
    Median:  {np.median(elat):.2f} ms

  Search only:
    Mean:    {slat.mean():.2f} ms
    Median:  {np.median(slat):.2f} ms
================================================
""")

# ===================== SAVE RESULTS =====================
results_summary = {
    "vector_store":      "Qdrant-InMemory",
    "chunk_size":        chunk_size,
    "chunk_overlap":     chunk_overlap,
    "total_docs":        total_docs,
    "total_chunks":      len(all_chunks_text),
    "total_queries":     total_queries,
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "latency_mean_ms":   round(lat.mean(), 2),
    "latency_p95_ms":    round(np.percentile(lat, 95), 2),
    "latency_p99_ms":    round(np.percentile(lat, 99), 2),
    "encode_mean_ms":    round(elat.mean(), 2),
    "search_mean_ms":    round(slat.mean(), 2),
}

print(pd.DataFrame([results_summary]).to_string(index=False))

Evaluating 140 queries...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API


==================== RESULTS ====================
  Total queries:  140
  Hit@5:          0.6000
  Hit@10:         0.6929

============== QUERY LATENCY (ms) ==============
  Total (encode + search):
    Mean:    717.64 ms
    Median:  686.53 ms
    P95:     881.78 ms
    P99:     924.52 ms

  Encode only:
    Mean:    28.59 ms
    Median:  25.83 ms

  Search only:
    Mean:    689.06 ms
    Median:  659.74 ms

   vector_store  chunk_size  chunk_overlap  total_docs  total_chunks  total_queries  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms  encode_mean_ms  search_mean_ms
Qdrant-InMemory         600             90       68611        165125            140    0.6  0.6929           717.64          881.78          924.52           28.59          689.06
